In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Data/Telco_churn.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
df.info()
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [10]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df["TotalCharges"].isnull().sum())   # expect 11
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print(df["TotalCharges"].dtype)            # expect float64
print(df.isnull().sum().sum())             # expect 0

11
float64
0


In [11]:
# SeniorCitizen is 0/1, convert to Yes/No to match other columns
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# "No internet service" / "No phone service" mean the same as "No" for these columns
cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for c in cols:
    df[c] = df[c].replace({"No internet service": "No", "No phone service": "No"})

# Target column as 0/1 for modelling later
df["Churn_flag"] = df["Churn"].map({"Yes": 1, "No": 0})

In [12]:
# 1. Tenure group (same bands as your SQL Q3)
df["tenure_group"] = pd.cut(df["tenure"],
                            bins=[-1, 12, 24, 48, 100],
                            labels=["0-12 months", "13-24 months", "25-48 months", "48+ months"])

# 2. Monthly charge category
df["charge_category"] = pd.qcut(df["MonthlyCharges"], q=3,
                                labels=["Low", "Medium", "High"])

# 3. Number of add-on services used
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

# 4. Average spend per month of tenure (0 for brand-new customers)
df["avg_monthly_spend"] = np.where(df["tenure"] > 0,
                                   df["TotalCharges"] / df["tenure"],
                                   df["MonthlyCharges"]).round(2)

# 5. Auto-pay flag
df["auto_pay"] = df["PaymentMethod"].str.contains("automatic").map({True: "Yes", False: "No"})

In [13]:
print(df["Churn_flag"].mean().round(4) * 100)                       # should be 26.54
print(df.groupby("tenure_group", observed=True)["Churn_flag"].mean().mul(100).round(2))
print(df.groupby("Contract")["Churn_flag"].mean().mul(100).round(2))

26.540000000000003
tenure_group
0-12 months     47.44
13-24 months    28.71
25-48 months    20.39
48+ months       9.51
Name: Churn_flag, dtype: float64
Contract
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: Churn_flag, dtype: float64


In [14]:
df.to_csv("../Data/telco_churn_cleaned.csv", index=False)
print(df.shape)

(7043, 27)
